In [15]:
import os
from dataclasses import dataclass
from typing import List
from scipy.stats import norm

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import sys




# Market Data

In [4]:
DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

tickers = ["SPY", "QQQ", "GLD", "BND", "VTI"]

data = yf.download(
    tickers=tickers,
    start="2015-01-01",
    end="2026-07-01",
    auto_adjust=True,
    progress=False
)

# Extract adjusted close prices
if isinstance(data.columns, pd.MultiIndex):
    prices = data["Close"].copy()
else:
    prices = data[["Close"]].copy()
    prices.columns = tickers

# Clean missing values
prices = prices.ffill().dropna()

# Calculate log returns
returns = np.log(prices / prices.shift(1)).dropna()

# Create simple portfolio
portfolio = pd.DataFrame({
    "Ticker": tickers,
    "Quantity": [100, 80, 50, 120, 90]
})

latest_prices = prices.iloc[-1]
portfolio["Latest Price"] = portfolio["Ticker"].map(latest_prices)
portfolio["Market Value"] = portfolio["Quantity"] * portfolio["Latest Price"]

# Save files
prices_path = f"{DATA_DIR}/prices.csv"
returns_path = f"{DATA_DIR}/returns.csv"
portfolio_path = f"{DATA_DIR}/portfolio.csv"

prices.to_csv(prices_path)
returns.to_csv(returns_path)
portfolio.to_csv(portfolio_path, index=False)

print("Files saved successfully:")
print(prices_path)
print(returns_path)
print(portfolio_path)

print("\nPortfolio:")
display(portfolio)

Files saved successfully:
/content/data/prices.csv
/content/data/returns.csv
/content/data/portfolio.csv

Portfolio:


,Ticker,Quantity,Latest Price,Market Value
0,SPY,100,746.770020,74677.001953
1,QQQ,80,736.400024,58912.001953
2,GLD,50,368.380005,18419.000244
3,BND,120,73.166000,8779.920044
4,VTI,90,370.040009,33303.600769


In [6]:
DATA_DIR = "/content/data"

prices = pd.read_csv(f"{DATA_DIR}/prices.csv", index_col=0, parse_dates=True)
returns = pd.read_csv(f"{DATA_DIR}/returns.csv", index_col=0, parse_dates=True)
portfolio = pd.read_csv(f"{DATA_DIR}/portfolio.csv")

print("Prices:")
display(prices.tail())

print("Returns:")
display(returns.tail())

print("Portfolio:")
display(portfolio)

Prices:


,BND,GLD,QQQ,SPY,VTI
Date,,,,,
2026-06-24,73.305534,365.920013,710.619995,733.239990,362.606934
2026-06-25,73.355370,369.459991,716.380005,734.299988,362.936005
2026-06-26,73.425133,373.630005,706.520020,728.989990,362.220001
2026-06-29,73.465004,368.579987,724.080017,741.000000,367.119995
2026-06-30,73.166000,368.380005,736.400024,746.770020,370.040009


Returns:


,BND,GLD,QQQ,SPY,VTI
Date,,,,,
2026-06-24,0.004497,-0.030679,-0.004255,-0.000464,-0.000138
2026-06-25,0.000680,0.009628,0.008073,0.001445,0.000907
2026-06-26,0.000951,0.011224,-0.013859,-0.007258,-0.001975
2026-06-29,0.000543,-0.013608,0.024550,0.016341,0.013437
2026-06-30,-0.004078,-0.000543,0.016872,0.007757,0.007922


Portfolio:


,Ticker,Quantity,Latest Price,Market Value
0,SPY,100,746.770020,74677.001953
1,QQQ,80,736.400024,58912.001953
2,GLD,50,368.380005,18419.000244
3,BND,120,73.166000,8779.920044
4,VTI,90,370.040009,33303.600769


Module 2 — Portfolio Pricing Engine

Create a file containing the functions to calculate:
Current Market Value;
Daily Portfolio Value;
Daily PnL;
Daily Portfolio Return;
Cumulative Return;
Portfolio Weights.

In [8]:


PROJECT_DIR = "/content/drive/MyDrive/market_risk_project"
SRC_DIR = f"{PROJECT_DIR}/src"
DATA_DIR = f"{PROJECT_DIR}/data"

os.makedirs(SRC_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Source code directory:", SRC_DIR)
print("Data directory:", DATA_DIR)

!cp /content/data/prices.csv /content/drive/MyDrive/market_risk_project/data/
!cp /content/data/returns.csv /content/drive/MyDrive/market_risk_project/data/
!cp /content/data/portfolio.csv /content/drive/MyDrive/market_risk_project/data/

Project directory: /content/drive/MyDrive/market_risk_project
Source code directory: /content/drive/MyDrive/market_risk_project/src
Data directory: /content/drive/MyDrive/market_risk_project/data


In [ ]:
!ls /content/drive/MyDrive/market_risk_project/data

# Portfolio Pricing Engin

In [9]:
%%writefile /content/drive/MyDrive/market_risk_project/src/pricing_engine.py

import pandas as pd
import numpy as np


class PortfolioPricingEngine:
    """
    Module 2: Portfolio Pricing Engine

    This class calculates:
    - Daily position values
    - Total portfolio market value
    - Current market value
    - Daily PnL
    - Daily portfolio returns
    - Cumulative return
    - Current portfolio weights
    """

    def __init__(self, prices: pd.DataFrame, portfolio: pd.DataFrame):
        self.prices = prices.copy()
        self.portfolio = portfolio.copy()

        self.tickers = self.portfolio["Ticker"].tolist()
        self.quantities = self.portfolio.set_index("Ticker")["Quantity"]

        self._validate_inputs()

    def _validate_inputs(self):
        """
        Check that every portfolio ticker exists in the price data.
        """

        missing_tickers = [
            ticker for ticker in self.tickers
            if ticker not in self.prices.columns
        ]

        if missing_tickers:
            raise ValueError(f"Missing price data for: {missing_tickers}")

    def calculate_daily_position_values(self) -> pd.DataFrame:
        """
        Calculate daily market value of each position.

        Position value = price × quantity
        """

        position_values = self.prices[self.tickers].multiply(
            self.quantities,
            axis=1
        )

        return position_values

    def calculate_daily_portfolio_value(self) -> pd.Series:
        """
        Calculate total portfolio value for each day.

        Portfolio value = sum of all position values
        """

        position_values = self.calculate_daily_position_values()
        portfolio_value = position_values.sum(axis=1)

        return portfolio_value

    def calculate_current_market_value(self) -> float:
        """
        Calculate latest total portfolio market value.
        """

        portfolio_value = self.calculate_daily_portfolio_value()
        current_market_value = portfolio_value.iloc[-1]

        return current_market_value

    def calculate_daily_pnl(self) -> pd.Series:
        """
        Calculate daily profit and loss.

        Daily PnL = today's portfolio value - yesterday's portfolio value
        """

        portfolio_value = self.calculate_daily_portfolio_value()
        daily_pnl = portfolio_value.diff().dropna()

        return daily_pnl

    def calculate_portfolio_returns(self) -> pd.Series:
        """
        Calculate daily portfolio returns.

        Portfolio return = today's portfolio value / yesterday's portfolio value - 1
        """

        portfolio_value = self.calculate_daily_portfolio_value()
        portfolio_returns = portfolio_value.pct_change().dropna()

        return portfolio_returns

    def calculate_cumulative_return(self) -> pd.Series:
        """
        Calculate cumulative portfolio return.

        Cumulative return = portfolio value today / initial portfolio value - 1
        """

        portfolio_value = self.calculate_daily_portfolio_value()
        cumulative_return = portfolio_value / portfolio_value.iloc[0] - 1

        return cumulative_return

    def calculate_current_weights(self) -> pd.DataFrame:
        """
        Calculate current portfolio weights.

        Weight = position market value / total portfolio market value
        """

        latest_prices = self.prices[self.tickers].iloc[-1]
        current_position_values = latest_prices * self.quantities
        total_value = current_position_values.sum()

        weights = current_position_values / total_value

        weights_df = pd.DataFrame({
            "Ticker": weights.index,
            "Quantity": self.quantities.values,
            "Latest Price": latest_prices.values,
            "Market Value": current_position_values.values,
            "Weight": weights.values
        })

        return weights_df

    def run(self) -> dict:
        """
        Run full portfolio pricing engine.
        """

        position_values = self.calculate_daily_position_values()
        portfolio_value = self.calculate_daily_portfolio_value()
        current_market_value = self.calculate_current_market_value()
        daily_pnl = self.calculate_daily_pnl()
        portfolio_returns = self.calculate_portfolio_returns()
        cumulative_return = self.calculate_cumulative_return()
        weights = self.calculate_current_weights()

        return {
            "position_values": position_values,
            "portfolio_value": portfolio_value,
            "current_market_value": current_market_value,
            "daily_pnl": daily_pnl,
            "portfolio_returns": portfolio_returns,
            "cumulative_return": cumulative_return,
            "weights": weights
        }

Writing /content/drive/MyDrive/market_risk_project/src/pricing_engine.py


In [10]:
!ls /content/drive/MyDrive/market_risk_project/src

pricing_engine.py


# Market Risk Engine

In [ ]:
%%writefile /content/drive/MyDrive/market_risk_project/src/risk_engine.py

import pandas as pd
import numpy as np


class MarketRiskEngine:
    """
    Module 3: Market Risk Engine

    This class calculates traditional market risk metrics:
    - Asset volatility
    - Correlation matrix
    - Covariance matrix
    - Portfolio volatility
    - Rolling volatility
    - Portfolio return distribution statistics
    """

    def __init__(
        self,
        asset_returns: pd.DataFrame,
        portfolio_returns,
        weights: pd.DataFrame,
        trading_days: int = 252
    ):
        self.asset_returns = asset_returns.copy()
        self.trading_days = trading_days

        # Convert portfolio returns from DataFrame to Series if needed
        if isinstance(portfolio_returns, pd.DataFrame):
            self.portfolio_returns = portfolio_returns.iloc[:, 0].copy()
        else:
            self.portfolio_returns = portfolio_returns.copy()

        self.weights_df = weights.copy()
        self.tickers = self.weights_df["Ticker"].tolist()
        self.weight_vector = self.weights_df.set_index("Ticker")["Weight"]

        self._validate_inputs()

    def _validate_inputs(self):
        """
        Make sure all portfolio tickers exist in the asset return data.
        """

        missing_tickers = [
            ticker for ticker in self.tickers
            if ticker not in self.asset_returns.columns
        ]

        if missing_tickers:
            raise ValueError(f"Missing return data for: {missing_tickers}")

    def calculate_asset_volatility(self) -> pd.DataFrame:
        """
        Calculate daily and annualized volatility for each asset.

        Daily volatility = standard deviation of daily returns
        Annualized volatility = daily volatility × sqrt(252)
        """

        daily_vol = self.asset_returns[self.tickers].std()
        annualized_vol = daily_vol * np.sqrt(self.trading_days)

        vol_df = pd.DataFrame({
            "Ticker": daily_vol.index,
            "Daily Volatility": daily_vol.values,
            "Annualized Volatility": annualized_vol.values
        })

        return vol_df

    def calculate_correlation_matrix(self) -> pd.DataFrame:
        """
        Calculate the correlation matrix of asset returns.
        """

        correlation_matrix = self.asset_returns[self.tickers].corr()
        return correlation_matrix

    def calculate_covariance_matrix(self) -> pd.DataFrame:
        """
        Calculate the annualized covariance matrix.

        Annualized covariance = daily covariance × 252
        """

        daily_covariance = self.asset_returns[self.tickers].cov()
        annualized_covariance = daily_covariance * self.trading_days

        return annualized_covariance

    def calculate_portfolio_volatility(self) -> float:
        """
        Calculate annualized portfolio volatility.

        Portfolio variance = w.T @ covariance_matrix @ w
        Portfolio volatility = sqrt(portfolio variance)
        """

        covariance_matrix = self.calculate_covariance_matrix()

        # Match weight order to covariance matrix columns
        ordered_weights = self.weight_vector.loc[covariance_matrix.columns].values

        portfolio_variance = ordered_weights.T @ covariance_matrix.values @ ordered_weights
        portfolio_volatility = np.sqrt(portfolio_variance)

        return portfolio_volatility

    def calculate_rolling_volatility(self, window: int = 60) -> pd.Series:
        """
        Calculate rolling annualized portfolio volatility.

        Default window = 60 trading days.
        """

        rolling_volatility = (
            self.portfolio_returns
            .rolling(window=window)
            .std()
            * np.sqrt(self.trading_days)
        )

        return rolling_volatility.dropna()

    def calculate_return_distribution_stats(self) -> pd.DataFrame:
        """
        Calculate descriptive statistics for portfolio returns.
        """

        stats = {
            "Mean Daily Return": self.portfolio_returns.mean(),
            "Daily Volatility": self.portfolio_returns.std(),
            "Annualized Return": self.portfolio_returns.mean() * self.trading_days,
            "Annualized Volatility": self.portfolio_returns.std() * np.sqrt(self.trading_days),
            "Minimum Daily Return": self.portfolio_returns.min(),
            "Maximum Daily Return": self.portfolio_returns.max(),
            "Skewness": self.portfolio_returns.skew(),
            "Kurtosis": self.portfolio_returns.kurtosis()
        }

        stats_df = pd.DataFrame(stats, index=["Portfolio"]).T
        stats_df.columns = ["Value"]

        return stats_df

    def run(self) -> dict:
        """
        Run the full market risk engine.
        """

        asset_volatility = self.calculate_asset_volatility()
        correlation_matrix = self.calculate_correlation_matrix()
        covariance_matrix = self.calculate_covariance_matrix()
        portfolio_volatility = self.calculate_portfolio_volatility()
        rolling_volatility = self.calculate_rolling_volatility(window=60)
        return_stats = self.calculate_return_distribution_stats()

        return {
            "asset_volatility": asset_volatility,
            "correlation_matrix": correlation_matrix,
            "covariance_matrix": covariance_matrix,
            "portfolio_volatility": portfolio_volatility,
            "rolling_volatility": rolling_volatility,
            "return_stats": return_stats
        }

# Historical VaR Engine

In [ ]:
%%writefile /content/drive/MyDrive/market_risk_project/src/var_engine.py

import pandas as pd
import numpy as np


class HistoricalVaREngine:
    """
    Module 4: Historical Value-at-Risk Engine

    This class calculates one-day Historical VaR using the left tail
    of historical portfolio returns.

    VaR is a one-tail downside risk measure.
    """

    def __init__(
        self,
        portfolio_returns,
        portfolio_value,
        confidence_levels: list = [0.95, 0.99]
    ):
        if isinstance(portfolio_returns, pd.DataFrame):
            self.portfolio_returns = portfolio_returns.iloc[:, 0].copy()
        else:
            self.portfolio_returns = portfolio_returns.copy()

        if isinstance(portfolio_value, pd.DataFrame):
            self.portfolio_value = portfolio_value.iloc[:, 0].copy()
        else:
            self.portfolio_value = portfolio_value.copy()

        self.confidence_levels = confidence_levels
        self.current_portfolio_value = self.portfolio_value.iloc[-1]

        self._validate_inputs()

    def _validate_inputs(self):
        if self.portfolio_returns.empty:
            raise ValueError("Portfolio returns are empty.")

        if self.portfolio_value.empty:
            raise ValueError("Portfolio value series is empty.")

        if self.current_portfolio_value <= 0:
            raise ValueError("Current portfolio value must be positive.")

    def calculate_historical_var(self) -> pd.DataFrame:
        """
        Calculate one-tail Historical VaR.

        For 95% VaR:
        - Use the 5th percentile of historical portfolio returns.

        For 99% VaR:
        - Use the 1st percentile of historical portfolio returns.

        This is left-tail risk because we are looking at the most negative returns.
        """

        results = []

        for confidence_level in self.confidence_levels:
            tail_probability = 1 - confidence_level

            # One-tail left-tail return threshold
            var_return = self.portfolio_returns.quantile(tail_probability)

            # Convert negative return into positive dollar loss
            var_amount = -var_return * self.current_portfolio_value

            results.append({
                "Confidence Level": confidence_level,
                "Tail Probability": tail_probability,
                "VaR Return Threshold": var_return,
                "VaR Amount": var_amount
            })

        return pd.DataFrame(results)

    def calculate_tail_observations(self, confidence_level: float = 0.95) -> pd.DataFrame:
        """
        Return only the left-tail observations used to understand VaR.

        Example:
        For 95% VaR, this returns returns below the 5th percentile.
        """

        tail_probability = 1 - confidence_level
        var_return = self.portfolio_returns.quantile(tail_probability)

        tail_returns = self.portfolio_returns[
            self.portfolio_returns <= var_return
        ].sort_values()

        tail_df = pd.DataFrame({
            "Portfolio Return": tail_returns,
            "Estimated Loss": -tail_returns * self.current_portfolio_value
        })

        return tail_df

    def calculate_worst_historical_days(self, n: int = 10) -> pd.DataFrame:
        """
        Show the worst historical portfolio return days.
        These are the most negative returns, so this is also left-tail.
        """

        worst_returns = self.portfolio_returns.sort_values().head(n)

        worst_days = pd.DataFrame({
            "Portfolio Return": worst_returns,
            "Estimated Loss": -worst_returns * self.current_portfolio_value
        })

        return worst_days

    def calculate_latest_pnl(self, n: int = 10) -> pd.DataFrame:
        """
        Show recent daily portfolio return and estimated PnL.

        This is NOT VaR.
        It is only for checking recent portfolio movement.
        """

        latest_returns = self.portfolio_returns.tail(n)

        latest_pnl = pd.DataFrame({
            "Portfolio Return": latest_returns,
            "Estimated PnL": latest_returns * self.current_portfolio_value
        })

        return latest_pnl

    def run(self) -> dict:
        var_table = self.calculate_historical_var()
        worst_days = self.calculate_worst_historical_days(n=10)
        tail_95 = self.calculate_tail_observations(confidence_level=0.95)
        tail_99 = self.calculate_tail_observations(confidence_level=0.99)
        latest_pnl = self.calculate_latest_pnl(n=10)

        return {
            "var_table": var_table,
            "worst_days": worst_days,
            "tail_95": tail_95,
            "tail_99": tail_99,
            "latest_pnl": latest_pnl,
            "current_portfolio_value": self.current_portfolio_value
        }

# black_scholes

In [14]:
%%writefile /content/drive/MyDrive/market_risk_project/src/black_scholes.py

import numpy as np
import pandas as pd
from scipy.stats import norm


class BlackScholesOption:
    """
    Black-Scholes model for European call option pricing and Greeks.
    """

    def __init__(self, S, K, T, r, sigma):
        self.S = float(S)
        self.K = float(K)
        self.T = float(T)
        self.r = float(r)
        self.sigma = float(sigma)

    def d1(self):
        return (
            np.log(self.S / self.K)
            + (self.r + 0.5 * self.sigma ** 2) * self.T
        ) / (self.sigma * np.sqrt(self.T))

    def d2(self):
        return self.d1() - self.sigma * np.sqrt(self.T)

    def call_price(self):
        d1 = self.d1()
        d2 = self.d2()

        return (
            self.S * norm.cdf(d1)
            - self.K * np.exp(-self.r * self.T) * norm.cdf(d2)
        )

    def delta(self):
        return norm.cdf(self.d1())

    def gamma(self):
        return norm.pdf(self.d1()) / (self.S * self.sigma * np.sqrt(self.T))

    def vega(self):
        # Per 1% change in volatility
        return self.S * norm.pdf(self.d1()) * np.sqrt(self.T) / 100

    def theta(self):
        # Per calendar day
        d1 = self.d1()
        d2 = self.d2()

        theta_annual = (
            -self.S * norm.pdf(d1) * self.sigma / (2 * np.sqrt(self.T))
            - self.r * self.K * np.exp(-self.r * self.T) * norm.cdf(d2)
        )

        return theta_annual / 365

    def rho(self):
        # Per 1% change in interest rate
        d2 = self.d2()

        return self.K * self.T * np.exp(-self.r * self.T) * norm.cdf(d2) / 100

    def summary(self):
        result = {
            "Option Price": self.call_price(),
            "Delta": self.delta(),
            "Gamma": self.gamma(),
            "Vega": self.vega(),
            "Theta": self.theta(),
            "Rho": self.rho()
        }

        return pd.DataFrame(result, index=["European Call"]).T

Overwriting /content/drive/MyDrive/market_risk_project/src/black_scholes.py
